In [1]:
import sys
import torch
import os
# sys.path.append('/home/campus.ncl.ac.uk/c4071391/Projects/DPVO/dpvo')
import dpvo
import collections
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
import numpy as np
import pandas as pd
verbose = False

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.set_printoptions(precision=10, sci_mode=False, linewidth=200)


In [2]:
# load onnx models
so = ort.SessionOptions()
so.log_severity_level = 2  # 0 = verbose

onnx_dir = '/home/campus.ncl.ac.uk/c4071391/Projects/DPVO/andy/onnx'
print(f'Loading onnx update modular model')
module_paths = {'patchify': os.path.join(onnx_dir, "patchify.onnx"),
                'update': os.path.join(onnx_dir, "update.onnx"),
                'corr': os.path.join(onnx_dir, "corr.onnx"),
                'norm': os.path.join(onnx_dir, "norm.onnx"),
                'c1': os.path.join(onnx_dir, "c1.onnx"),
                'c2': os.path.join(onnx_dir, "c2.onnx"),
                'agg_kk': os.path.join(onnx_dir, "agg_kk.onnx"),
                'agg_ij': os.path.join(onnx_dir, "agg_ij.onnx"),
                'gru': os.path.join(onnx_dir, "gru.onnx"),
                'w': os.path.join(onnx_dir, "w.onnx"),
                'd': os.path.join(onnx_dir, "d.onnx")}
module_sessions = {}
for name, module_path in module_paths.items():
    if not os.path.isfile(module_path):
        raise FileNotFoundError(f"ONNX encoder file not found in {onnx_dir} for {name}. Run andy/onnx_conversion.ipynb first.")
    onnx_dir_str = os.path.normpath(str(onnx_dir))
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    # providers = ["CUDAExecutionProvider"]
    module_sessions[name] = ort.InferenceSession(module_path, sess_options=so, providers=providers)
    if verbose:
        print("=== inputs ===")
        for i in module_sessions[name].get_inputs():
            print(i.name, i.shape, i.type)
        print("=== outputs ===")
        for o in module_sessions[name].get_outputs():
            print(o.name, o.shape, o.type)
    if verbose: print(f'Onnx {name} module loaded: {module_sessions[name]}')

Loading onnx update modular model


2026-04-03 16:50:03.470018930 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 29 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2026-04-03 16:50:03.527962792 [W:onnxruntime:Default, scatter_nd.h:51 ScatterNDWithAtomicReduction] ScatterND with reduction=='none' only guarantees to be correct if indices are not duplicated.
2026-04-03 16:50:03.527979865 [W:onnxruntime:Default, scatter_nd.h:51 ScatterNDWithAtomicReduction] ScatterND with reduction=='none' only guarantees to be correct if indices are not duplicated.


In [3]:
# load pytorch models
project_root = '/home/campus.ncl.ac.uk/c4071391/Projects/DPVO/'

from dpvo.net import VONet

# --- 1. Load the DPVO PyTorch Model ---
# Weights path: project root dpvo.pth or andy/dpvo.pth
pth_model_path = os.path.join(project_root, "dpvo.pth")
if not os.path.isfile(pth_model_path):
    pth_model_path = os.path.join(project_root, "andy", "dpvo.pth")
assert os.path.isfile(pth_model_path), f"Checkpoint not found: {pth_model_path}"

export_device = torch.device("cuda")

model = VONet()

ckpt = torch.load(pth_model_path, map_location="cuda", weights_only=True)
state_dict = ckpt if isinstance(ckpt, dict) and "state_dict" not in ckpt else ckpt.get("state_dict", ckpt)

# Strip DataParallel prefix; drop update.lmbda if present (DPVO load_weights does this)
new_state_dict = collections.OrderedDict()
for k, v in state_dict.items():
    if "update.lmbda" in k:
        print(k)
        continue
    new_state_dict[k.replace("module.", "")] = v
    print(k)

missing, unexpected = model.load_state_dict(new_state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model.eval().to(export_device)



module.patchify.fnet.conv1.weight
module.patchify.fnet.conv1.bias
module.patchify.fnet.layer1.0.conv1.weight
module.patchify.fnet.layer1.0.conv1.bias
module.patchify.fnet.layer1.0.conv2.weight
module.patchify.fnet.layer1.0.conv2.bias
module.patchify.fnet.layer1.1.conv1.weight
module.patchify.fnet.layer1.1.conv1.bias
module.patchify.fnet.layer1.1.conv2.weight
module.patchify.fnet.layer1.1.conv2.bias
module.patchify.fnet.layer2.0.conv1.weight
module.patchify.fnet.layer2.0.conv1.bias
module.patchify.fnet.layer2.0.conv2.weight
module.patchify.fnet.layer2.0.conv2.bias
module.patchify.fnet.layer2.0.downsample.0.weight
module.patchify.fnet.layer2.0.downsample.0.bias
module.patchify.fnet.layer2.1.conv1.weight
module.patchify.fnet.layer2.1.conv1.bias
module.patchify.fnet.layer2.1.conv2.weight
module.patchify.fnet.layer2.1.conv2.bias
module.patchify.fnet.conv2.weight
module.patchify.fnet.conv2.bias
module.patchify.inet.conv1.weight
module.patchify.inet.conv1.bias
module.patchify.inet.layer1.0.co

/home/campus.ncl.ac.uk/c4071391/miniconda3/envs/dpvo/lib/python3.9/site-packages/dpvo/net.py:187: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)


VONet(
  (patchify): Patchifier(
    (fnet): BasicEncoder4(
      (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (conv1): Conv2d(3, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
      (relu1): ReLU(inplace=True)
      (layer1): Sequential(
        (0): ResidualBlock(
          (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (relu): ReLU(inplace=True)
          (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
          (norm2): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        )
        (1): ResidualBlock(
          (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (relu): ReLU(inplace=True)
          

In [4]:
# load dummy_inputs:
INPUT_PAYLOAD_PATH = os.path.join('onnx', 'input_payload.pth')

device = 'cuda'

payload = torch.load(INPUT_PAYLOAD_PATH, map_location=device)
net = payload['net_in'].float()
ctx = payload['inp'].float()
corr = payload['corr'].float()
ii = payload['ii'].long()
jj = payload['jj'].long()
kk = payload['kk'].long()
E_real = int(net.shape[1])

B, E_real, D = net.shape
_, _, Cc = corr.shape


/tmp/ipykernel_199939/3654949096.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(INPUT_PAYLOAD_PATH, map_location=device)


In [5]:
# helper methods
def bind_torch_inputs(io_binding, inputs: dict, device="cuda", device_id=0):
    for name, tensor in inputs.items():
        if tensor is None:
            continue

        assert tensor.is_cuda, f"{name} must be on GPU for zero-copy"

        io_binding.bind_input(
            name=name,
            device_type=device,
            device_id=device_id,
            element_type=(
                np.float32 if tensor.dtype in (torch.float32, torch.float16)
                else np.int64
            ),
            shape=tuple(tensor.shape),
            buffer_ptr=tensor.data_ptr(),
        )

comparison_rows = []


def record_comparison_summary(model: str, max_abs_diff: float, allclose: bool) -> None:
    comparison_rows.append(
        {"model": model, "max_abs_diff": max_abs_diff, "allclose": allclose}
    )


def bind_torch_output(io_binding, name, tensor, device="cuda", device_id=0):
    io_binding.bind_output(
        name=name,
        device_type=device,
        device_id=device_id,
        element_type=(
            np.float32 if tensor.dtype in (torch.float32, torch.float16) else np.int64
        ),
        shape=tuple(tensor.shape),
        buffer_ptr=tensor.data_ptr(),
    )


def assert_close(name, a, b, atol=1e-8, rtol=1e-5):
    d = (a - b).abs().max().item()
    ok = torch.allclose(a, b, atol=atol, rtol=rtol)
    print(f"{name}: max_abs_diff={d:.3e}  allclose={ok}")
    return d, ok
    # assert ok, f"{name} mismatch"


In [18]:
#--------------------patchify----------------------
comparison_rows.clear()

# Match andy/onnx_conversion.ipynb: normalized images [B,N,3,H,W], patches_per_image=96
BATCH, NUM_FRAMES, C, H, W = 1, 1, 3, 480, 640
PATCHES_PER_IMAGE = 96

images_raw = (torch.rand(BATCH, NUM_FRAMES, C, H, W, device="cuda") * 255.0).to(torch.float32)
images = (2.0 * (images_raw / 255.0) - 0.5).to(torch.float32).contiguous()
ppi = torch.tensor(PATCHES_PER_IMAGE, dtype=torch.int64, device="cuda")

with torch.no_grad():
    pt_ref = model.patchify(images, patches_per_image=PATCHES_PER_IMAGE, centroid_sel_strat='RANDOM',return_color=True)
    print(f'pt_ref shape: {len(pt_ref)}')
onnx_fmap = torch.empty_like(pt_ref[0])
onnx_gmap = torch.empty_like(pt_ref[1])
onnx_imap = torch.empty_like(pt_ref[2])
onnx_patches = torch.empty_like(pt_ref[3])
onnx_index = torch.empty_like(pt_ref[4])
onnx_clr = torch.empty_like(pt_ref[5])

patchify_io = module_sessions["patchify"].io_binding()
bind_torch_inputs(patchify_io, {"images": images, "patches_per_image": ppi})
for name, t in [
    ("fmap", onnx_fmap),
    ("gmap", onnx_gmap),
    ("imap", onnx_imap),
    ("patches", onnx_patches),
    ("index", onnx_index),
    ("clr", onnx_clr),
]:
    bind_torch_output(patchify_io, name, t)

pytorch_results = []
onnx_results = []
iterations = 10
max_abs = 0.0
all_ok = True

for i in range(iterations):
    module_sessions["patchify"].run_with_iobinding(patchify_io)
    onnx_results.append(
        (onnx_fmap.clone(), onnx_gmap.clone(), onnx_imap.clone(), onnx_patches.clone(), onnx_index.clone(), onnx_clr.clone())
    )
    with torch.no_grad():
        f_pt, g_pt, i_pt, p_pt, idx_pt, c_pt = model.patchify(images, patches_per_image=PATCHES_PER_IMAGE, centroid_sel_strat='RANDOM', return_color=True)
    pytorch_results.append((f_pt, g_pt, i_pt, p_pt, idx_pt, c_pt))

    for sub, a, b in [
        ("patchify/fmap", onnx_fmap, f_pt),
        ("patchify/gmap", onnx_gmap, g_pt),
        ("patchify/imap", onnx_imap, i_pt),
        ("patchify/patches", onnx_patches, p_pt),
        ("patchify/clr", onnx_clr, c_pt),
    ]:
        d, ok = assert_close(sub, a, b)
        max_abs = max(max_abs, d)
        all_ok = all_ok and ok
    idx_equal = torch.equal(onnx_index, idx_pt)
    print(f"patchify/index equal: {idx_equal}")
    all_ok = all_ok and idx_equal
    if i == iterations - 1:
        print(f"Onnx fmap slice: {onnx_fmap.flatten()[:6]}")
        print(f"Pytorch fmap slice: {f_pt.flatten()[:6]}")

all_onnx_equal = all(
    all(torch.equal(onnx_results[0][j], t[j]) for j in range(6)) for t in onnx_results[1:]
)
all_pytorch_equal = all(
    all(torch.equal(pytorch_results[0][j], t[j]) for j in range(6)) for t in pytorch_results[1:]
)
print(f"All Onnx Results The Same? {all_onnx_equal}")
print(f"All Pytorch Results The Same? {all_pytorch_equal}")
record_comparison_summary("patchify", max_abs, all_ok)

pt_ref shape: 6
patchify/fmap: max_abs_diff=4.025e-04  allclose=False
patchify/gmap: max_abs_diff=1.349e+00  allclose=False
patchify/imap: max_abs_diff=1.405e+00  allclose=False
patchify/patches: max_abs_diff=1.330e+02  allclose=False
patchify/clr: max_abs_diff=1.957e+00  allclose=False
patchify/index equal: True
patchify/fmap: max_abs_diff=4.025e-04  allclose=False
patchify/gmap: max_abs_diff=1.503e+00  allclose=False
patchify/imap: max_abs_diff=2.071e+00  allclose=False
patchify/patches: max_abs_diff=1.330e+02  allclose=False
patchify/clr: max_abs_diff=1.905e+00  allclose=False
patchify/index equal: True
patchify/fmap: max_abs_diff=4.025e-04  allclose=False
patchify/gmap: max_abs_diff=1.611e+00  allclose=False
patchify/imap: max_abs_diff=2.090e+00  allclose=False
patchify/patches: max_abs_diff=1.390e+02  allclose=False
patchify/clr: max_abs_diff=1.920e+00  allclose=False
patchify/index equal: True
patchify/fmap: max_abs_diff=4.025e-04  allclose=False
patchify/gmap: max_abs_diff=1.460

In [7]:
#--------------------update (full graph)----------------------
net_u = net.to(torch.float32, copy=False).contiguous()
ctx_u = ctx.to(torch.float32, copy=False).contiguous()
corr_u = corr.to(torch.float32, copy=False).contiguous()
flow_u = torch.zeros(B, E_real, 2, device=net.device, dtype=torch.float32)
ii_u = ii.contiguous()
jj_u = jj.contiguous()
kk_u = kk.contiguous()

with torch.no_grad():
    net_pt, (delta_pt, weight_pt, _) = model.update(net_u, ctx_u, corr_u, flow_u, ii_u, jj_u, kk_u)

net_onnx = torch.empty_like(net_pt)
delta_onnx = torch.empty_like(delta_pt)
weight_onnx = torch.empty_like(weight_pt)

update_io = module_sessions["update"].io_binding()
bind_torch_inputs(
    update_io,
    {"net_in": net_u, "inp": ctx_u, "corr": corr_u, "ii": ii_u, "jj": jj_u, "kk": kk_u},
)
bind_torch_output(update_io, "net_out", net_onnx)
bind_torch_output(update_io, "delta_out", delta_onnx)
bind_torch_output(update_io, "weight_out", weight_onnx)

pytorch_results = []
onnx_results = []
iterations = 10
max_abs = 0.0
all_ok = True

for i in range(iterations):
    module_sessions["update"].run_with_iobinding(update_io)
    onnx_results.append((net_onnx.clone(), delta_onnx.clone(), weight_onnx.clone()))
    with torch.no_grad():
        n_pt, (d_pt, w_pt, _) = model.update(net_u, ctx_u, corr_u, flow_u, ii_u, jj_u, kk_u)
    pytorch_results.append((n_pt, d_pt, w_pt))

    for sub, a, b in [
        ("update/net_out", net_onnx, n_pt),
        ("update/delta_out", delta_onnx, d_pt),
        ("update/weight_out", weight_onnx, w_pt),
    ]:
        d, ok = assert_close(sub, a, b)
        max_abs = max(max_abs, d)
        all_ok = all_ok and ok
    if i == iterations - 1:
        print(f"Onnx net_out slice: {net_onnx.flatten()[:6]}")
        print(f"Pytorch net_out slice: {n_pt.flatten()[:6]}")

all_onnx_equal = all(
    all(torch.equal(onnx_results[0][j], t[j]) for j in range(3)) for t in onnx_results[1:]
)
all_pytorch_equal = all(
    all(torch.equal(pytorch_results[0][j], t[j]) for j in range(3)) for t in pytorch_results[1:]
)
print(f"All Onnx Results The Same? {all_onnx_equal}")
print(f"All Pytorch Results The Same? {all_pytorch_equal}")
record_comparison_summary("update", max_abs, all_ok)

2026-04-03 16:50:05.373065347 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,4992,384} for output /agg_kk/Identity_3_output_0
2026-04-03 16:50:05.406397917 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,629647,384} for output /agg_ij/Identity_3_output_0
2026-04-03 16:50:05.547403197 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,4992,384} for output /agg_kk/Identity_3_output_0
2026-04-03 16:50:05.547545183 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,629647,384} for output /agg_ij/Identity_3_output_0


update/net_out: max_abs_diff=1.829e+01  allclose=False
update/delta_out: max_abs_diff=6.959e+01  allclose=False
update/weight_out: max_abs_diff=9.842e-01  allclose=False
update/net_out: max_abs_diff=1.829e+01  allclose=False
update/delta_out: max_abs_diff=6.959e+01  allclose=False
update/weight_out: max_abs_diff=9.842e-01  allclose=False


2026-04-03 16:50:05.673908943 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,4992,384} for output /agg_kk/Identity_3_output_0
2026-04-03 16:50:05.674053110 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,629647,384} for output /agg_ij/Identity_3_output_0
2026-04-03 16:50:05.800633600 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,4992,384} for output /agg_kk/Identity_3_output_0
2026-04-03 16:50:05.800785444 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,629647,384} for output /agg_ij/Identity_3_output_0


update/net_out: max_abs_diff=1.829e+01  allclose=False
update/delta_out: max_abs_diff=6.959e+01  allclose=False
update/weight_out: max_abs_diff=9.842e-01  allclose=False
update/net_out: max_abs_diff=1.829e+01  allclose=False
update/delta_out: max_abs_diff=6.959e+01  allclose=False
update/weight_out: max_abs_diff=9.842e-01  allclose=False


2026-04-03 16:50:05.927769037 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,4992,384} for output /agg_kk/Identity_3_output_0
2026-04-03 16:50:05.927914987 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,629647,384} for output /agg_ij/Identity_3_output_0
2026-04-03 16:50:06.054417138 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,4992,384} for output /agg_kk/Identity_3_output_0
2026-04-03 16:50:06.054565481 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,629647,384} for output /agg_ij/Identity_3_output_0


update/net_out: max_abs_diff=1.829e+01  allclose=False
update/delta_out: max_abs_diff=6.959e+01  allclose=False
update/weight_out: max_abs_diff=9.842e-01  allclose=False
update/net_out: max_abs_diff=1.829e+01  allclose=False
update/delta_out: max_abs_diff=6.959e+01  allclose=False
update/weight_out: max_abs_diff=9.842e-01  allclose=False


2026-04-03 16:50:06.181157257 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,4992,384} for output /agg_kk/Identity_3_output_0
2026-04-03 16:50:06.181300852 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,629647,384} for output /agg_ij/Identity_3_output_0
2026-04-03 16:50:06.308249436 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,4992,384} for output /agg_kk/Identity_3_output_0
2026-04-03 16:50:06.308392298 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,629647,384} for output /agg_ij/Identity_3_output_0


update/net_out: max_abs_diff=1.829e+01  allclose=False
update/delta_out: max_abs_diff=6.959e+01  allclose=False
update/weight_out: max_abs_diff=9.842e-01  allclose=False
update/net_out: max_abs_diff=1.829e+01  allclose=False
update/delta_out: max_abs_diff=6.959e+01  allclose=False
update/weight_out: max_abs_diff=9.842e-01  allclose=False


2026-04-03 16:50:06.435205753 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,4992,384} for output /agg_kk/Identity_3_output_0
2026-04-03 16:50:06.435354507 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,629647,384} for output /agg_ij/Identity_3_output_0
2026-04-03 16:50:06.561840362 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,4992,384} for output /agg_kk/Identity_3_output_0
2026-04-03 16:50:06.561983303 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,629647,384} for output /agg_ij/Identity_3_output_0


update/net_out: max_abs_diff=1.829e+01  allclose=False
update/delta_out: max_abs_diff=6.959e+01  allclose=False
update/weight_out: max_abs_diff=9.842e-01  allclose=False
update/net_out: max_abs_diff=1.829e+01  allclose=False
update/delta_out: max_abs_diff=6.959e+01  allclose=False
update/weight_out: max_abs_diff=9.842e-01  allclose=False
Onnx net_out slice: tensor([-0.5298657417, -0.6737640500, -0.3260293603, -2.9785127640,  2.0681233406,  0.7460629940], device='cuda:0')
Pytorch net_out slice: tensor([-0.8588026762, -1.1686975956, -1.2513320446, -3.5807693005,  3.5467684269,  0.9793965220], device='cuda:0')
All Onnx Results The Same? True
All Pytorch Results The Same? False


In [8]:
#--------------------corr----------------------
corr_t = corr.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
corr_onnx_out = torch.empty((B, E_real, D), device=corr_t.device, dtype=torch.float32)
corr_io_binding = module_sessions['corr'].io_binding()
bind_torch_inputs(corr_io_binding, {'corr_input': corr_t})

corr_io_binding.bind_output(
    name='corr_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(corr_onnx_out.shape),
    buffer_ptr=corr_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['corr'].run_with_iobinding(corr_io_binding)
    onnx_results.append(corr_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        corr_pytorch_out = model.update.corr(corr_t)
    pytorch_results.append(corr_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("corr", corr_onnx_out, corr_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {corr_onnx_out}')
        print(f'Pytorch Output: {corr_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("corr", max_abs, all_ok)


corr: max_abs_diff=3.113e-03  allclose=False
corr: max_abs_diff=3.113e-03  allclose=False
corr: max_abs_diff=3.113e-03  allclose=False
corr: max_abs_diff=3.113e-03  allclose=False
corr: max_abs_diff=3.113e-03  allclose=False
corr: max_abs_diff=3.113e-03  allclose=False
corr: max_abs_diff=3.113e-03  allclose=False
corr: max_abs_diff=3.113e-03  allclose=False
corr: max_abs_diff=3.113e-03  allclose=False
corr: max_abs_diff=3.113e-03  allclose=False
Onnx Output: tensor([[[-1.3759294748,  0.2101907879,  1.6979593039,  ...,  1.0414646864, -0.3800020516,  1.9119734764],
         [-1.1505411863, -0.1133284420,  1.2951016426,  ...,  0.9306297302, -0.6189746261,  1.7636487484],
         [-1.5378347635,  0.1333798021,  2.0349087715,  ...,  0.8869853616, -0.6579807401,  1.3073196411],
         ...,
         [-0.3975575268,  1.4375818968, -0.1108961031,  ..., -0.3680568933,  1.6350829601,  1.4819345474],
         [-0.5343906283,  1.4766603708,  0.0073695853,  ..., -0.4587374926,  1.5548743010,  1.1

In [9]:
#--------------------norm----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
norm_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
norm_io_binding = module_sessions['norm'].io_binding()
bind_torch_inputs(norm_io_binding, {'net_input': net_t})

norm_io_binding.bind_output(
    name='norm_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(norm_onnx_out.shape),
    buffer_ptr=norm_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['norm'].run_with_iobinding(norm_io_binding)
    onnx_results.append(norm_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        norm_pytorch_out = model.update.norm(net_t)
    pytorch_results.append(norm_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("norm", norm_onnx_out, norm_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {norm_onnx_out}')
        print(f'Pytorch Output: {norm_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("norm", max_abs, all_ok)


norm: max_abs_diff=1.907e-06  allclose=False
norm: max_abs_diff=1.907e-06  allclose=False
norm: max_abs_diff=1.907e-06  allclose=False
norm: max_abs_diff=1.907e-06  allclose=False
norm: max_abs_diff=1.907e-06  allclose=False
norm: max_abs_diff=1.907e-06  allclose=False
norm: max_abs_diff=1.907e-06  allclose=False
norm: max_abs_diff=1.907e-06  allclose=False
norm: max_abs_diff=1.907e-06  allclose=False
norm: max_abs_diff=1.907e-06  allclose=False
Onnx Output: tensor([[[ 0.1277522743,  0.2157948911, -0.0416595452,  ..., -0.5235769153, -1.2212355137,  0.0847051442],
         [ 0.2564165592,  0.1029407606, -0.0902231336,  ..., -0.6846762896, -1.5460010767, -0.1938833743],
         [ 0.2689175308,  0.1163040176,  0.0228694417,  ..., -0.5182351470, -1.1262376308,  0.1859927624],
         ...,
         [ 0.5323991179,  0.2948898971,  0.1509808302,  ...,  0.2631749213, -0.9920817018,  0.2526425123],
         [ 0.6043900251,  0.2637344301, -0.0741459504,  ...,  0.3061308265, -0.8952465057,  0.1

In [10]:
#--------------------c1----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
c1_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
c1_io_binding = module_sessions['c1'].io_binding()
bind_torch_inputs(c1_io_binding, {'c1_input': net_t})

c1_io_binding.bind_output(
    name='c1_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(c1_onnx_out.shape),
    buffer_ptr=c1_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['c1'].run_with_iobinding(c1_io_binding)
    onnx_results.append(c1_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        c1_pytorch_out = model.update.c1(net_t)
    pytorch_results.append(c1_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("c1", c1_onnx_out, c1_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {c1_onnx_out}')
        print(f'Pytorch Output: {c1_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("c1", max_abs, all_ok)


c1: max_abs_diff=5.961e-03  allclose=False
c1: max_abs_diff=5.961e-03  allclose=False
c1: max_abs_diff=5.961e-03  allclose=False
c1: max_abs_diff=5.961e-03  allclose=False
c1: max_abs_diff=5.961e-03  allclose=False
c1: max_abs_diff=5.961e-03  allclose=False
c1: max_abs_diff=5.961e-03  allclose=False
c1: max_abs_diff=5.961e-03  allclose=False
c1: max_abs_diff=5.961e-03  allclose=False
c1: max_abs_diff=5.961e-03  allclose=False
Onnx Output: tensor([[[ 7.6210956573,  0.1458067894,  4.6569042206,  ...,  8.6765241623,  4.5624322891, -2.4694890976],
         [ 7.4967727661, -0.7892264128,  6.7043070793,  ...,  8.2513122559,  4.2119865417, -0.2687870860],
         [ 8.8561849594, -0.0209512357,  5.1722202301,  ...,  9.2553930283,  4.3180112839, -2.3467407227],
         ...,
         [ 8.5052566528,  2.0957293510,  4.8563919067,  ...,  7.3910155296,  5.6701269150, -2.7637631893],
         [ 8.7628021240,  1.2060990334,  5.4674453735,  ...,  6.2234449387,  5.2130174637, -2.5517382622],
        

In [11]:
#--------------------c2----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
c2_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
c2_io_binding = module_sessions['c2'].io_binding()
bind_torch_inputs(c2_io_binding, {'c2_input': net_t})

c2_io_binding.bind_output(
    name='c2_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(c2_onnx_out.shape),
    buffer_ptr=c2_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['c2'].run_with_iobinding(c2_io_binding)
    onnx_results.append(c2_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        c2_pytorch_out = model.update.c2(net_t)
    pytorch_results.append(c2_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("c2", c2_onnx_out, c2_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {c2_onnx_out}')
        print(f'Pytorch Output: {c2_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("c2", max_abs, all_ok)


c2: max_abs_diff=5.734e-03  allclose=False
c2: max_abs_diff=5.734e-03  allclose=False
c2: max_abs_diff=5.734e-03  allclose=False
c2: max_abs_diff=5.734e-03  allclose=False
c2: max_abs_diff=5.734e-03  allclose=False
c2: max_abs_diff=5.734e-03  allclose=False
c2: max_abs_diff=5.734e-03  allclose=False
c2: max_abs_diff=5.734e-03  allclose=False
c2: max_abs_diff=5.734e-03  allclose=False
c2: max_abs_diff=5.734e-03  allclose=False
Onnx Output: tensor([[[ 2.3358516693, -1.3911734819,  6.2402515411,  ..., -1.8425421715,  0.1222885996,  3.3621940613],
         [ 2.8835513592, -1.2499752045,  6.2487883568,  ..., -0.3890676796,  0.8496623635,  3.2934710979],
         [ 2.4878416061, -1.7489466667,  6.9968986511,  ..., -1.9719860554, -0.1308711618,  3.7805306911],
         ...,
         [ 2.1441261768, -0.7607700229,  6.3107223511,  ..., -2.0373253822, -0.7279564738,  3.6266379356],
         [ 2.2204816341, -0.2550342977,  6.1796665192,  ..., -2.0914835930, -0.7271148562,  4.8748488426],
        

In [12]:
#--------------------agg_kk----------------------

net_t = net.to(torch.float32, copy=False).contiguous()
_, jx_kk = torch.unique(kk, return_inverse=True)

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
agg_kk_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
agg_kk_io_binding = module_sessions['agg_kk'].io_binding()
bind_torch_inputs(agg_kk_io_binding, {'agg_kk_input': net_t, 'jx_input': jx_kk})

agg_kk_io_binding.bind_output(
    name='agg_kk_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(agg_kk_onnx_out.shape),
    buffer_ptr=agg_kk_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['agg_kk'].run_with_iobinding(agg_kk_io_binding)
    onnx_results.append(agg_kk_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        agg_kk_pytorch_out = model.update.agg_kk(net_t, jx_kk)
    pytorch_results.append(agg_kk_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("agg_kk", agg_kk_onnx_out, agg_kk_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {agg_kk_onnx_out}')
        print(f'Pytorch Output: {agg_kk_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("agg_kk", max_abs, all_ok)


agg_kk: max_abs_diff=3.576e-03  allclose=False
agg_kk: max_abs_diff=3.576e-03  allclose=False
agg_kk: max_abs_diff=3.576e-03  allclose=False
agg_kk: max_abs_diff=3.576e-03  allclose=False
agg_kk: max_abs_diff=3.575e-03  allclose=False


2026-04-03 16:50:07.135405939 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2208,384} for output /Identity_3_output_0
2026-04-03 16:50:07.164681787 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2208,384} for output /Identity_3_output_0
2026-04-03 16:50:07.186470073 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2208,384} for output /Identity_3_output_0
2026-04-03 16:50:07.208158827 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2208,384} for output /Identity_3_output_0
2026-04-03 16:50:07.230343189 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2208,384} for output /Identity_3_output_0
2026-

agg_kk: max_abs_diff=3.576e-03  allclose=False
agg_kk: max_abs_diff=3.576e-03  allclose=False
agg_kk: max_abs_diff=3.576e-03  allclose=False
agg_kk: max_abs_diff=3.576e-03  allclose=False
agg_kk: max_abs_diff=3.576e-03  allclose=False
Onnx Output: tensor([[[ 1.7797453403,  0.5458849072,  1.7666226625,  ...,  0.1485814154, -0.0279466640,  2.0222949982],
         [ 1.7797453403,  0.5458849072,  1.7666226625,  ...,  0.1485814154, -0.0279466640,  2.0222949982],
         [ 1.7797453403,  0.5458849072,  1.7666226625,  ...,  0.1485814154, -0.0279466640,  2.0222949982],
         ...,
         [ 1.1944727898,  1.3502805233,  1.9770423174,  ...,  0.1766078770, -0.7575462461,  1.4769445658],
         [ 1.1944727898,  1.3502805233,  1.9770423174,  ...,  0.1766078770, -0.7575462461,  1.4769445658],
         [ 1.1944727898,  1.3502805233,  1.9770423174,  ...,  0.1766078770, -0.7575462461,  1.4769445658]]], device='cuda:0')
Pytorch Output: tensor([[[ 1.7801506519,  0.5453713536,  1.7670669556,  ..., 

2026-04-03 16:50:07.340424355 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2208,384} for output /Identity_3_output_0


In [13]:
#--------------------agg_ij----------------------

net_t = net.to(torch.float32, copy=False).contiguous()
iijj = ii * 12345 + jj
_, jx_ij = torch.unique(iijj, return_inverse=True)

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
agg_ij_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
agg_ij_io_binding = module_sessions['agg_ij'].io_binding()
bind_torch_inputs(agg_ij_io_binding, {'agg_ij_input': net_t, 'jx_input': jx_ij})

agg_ij_io_binding.bind_output(
    name='agg_ij_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(agg_ij_onnx_out.shape),
    buffer_ptr=agg_ij_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['agg_ij'].run_with_iobinding(agg_ij_io_binding)
    onnx_results.append(agg_ij_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        agg_ij_pytorch_out = model.update.agg_ij(net_t, jx_ij)
    pytorch_results.append(agg_ij_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("agg_ij", agg_ij_onnx_out, agg_ij_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {agg_ij_onnx_out}')
        print(f'Pytorch Output: {agg_ij_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("agg_ij", max_abs, all_ok)


agg_ij: max_abs_diff=2.454e-03  allclose=False
agg_ij: max_abs_diff=2.454e-03  allclose=False
agg_ij: max_abs_diff=2.454e-03  allclose=False


2026-04-03 16:50:07.392812112 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,466,384} for output /Identity_3_output_0
2026-04-03 16:50:07.421677183 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,466,384} for output /Identity_3_output_0
2026-04-03 16:50:07.443571842 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,466,384} for output /Identity_3_output_0
2026-04-03 16:50:07.465290226 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,466,384} for output /Identity_3_output_0
2026-04-03 16:50:07.487180151 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,466,384} for output /Identity_3_output_0
2026-04-03

agg_ij: max_abs_diff=2.454e-03  allclose=False
agg_ij: max_abs_diff=2.454e-03  allclose=False
agg_ij: max_abs_diff=2.454e-03  allclose=False
agg_ij: max_abs_diff=2.454e-03  allclose=False


2026-04-03 16:50:07.552493479 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,466,384} for output /Identity_3_output_0
2026-04-03 16:50:07.574380501 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,466,384} for output /Identity_3_output_0


agg_ij: max_abs_diff=2.454e-03  allclose=False
agg_ij: max_abs_diff=2.454e-03  allclose=False
agg_ij: max_abs_diff=2.454e-03  allclose=False
Onnx Output: tensor([[[    -0.7766404152,      0.1817910820,     -0.0824853033,  ...,      3.2398760319,     -0.4508380294,     -1.6119686365],
         [    -0.6937636137,      0.0295198467,     -0.4620892406,  ...,      3.1136968136,     -0.5777982473,     -1.8850411177],
         [    -0.8924664855,      0.0032469928,     -0.1767735034,  ...,      2.8846316338,     -0.5928950310,     -1.6772286892],
         ...,
         [     1.3317070007,     -1.6990215778,      1.4666155577,  ...,      2.3558781147,     -1.9058294296,     -3.2204802036],
         [     1.7115695477,     -1.6267153025,      1.9588128328,  ...,      1.9416389465,     -1.8557963371,     -3.6038961411],
         [     1.6105701923,     -1.7312403917,      2.1597964764,  ...,      1.7702598572,     -2.1333749294,     -3.6792786121]]], device='cuda:0')
Pytorch Output: tensor([[[ 

2026-04-03 16:50:07.596166527 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,466,384} for output /Identity_3_output_0


In [14]:
#--------------------gru----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
gru_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
gru_io_binding = module_sessions['gru'].io_binding()
bind_torch_inputs(gru_io_binding, {'gru_input': net_t})

gru_io_binding.bind_output(
    name='net_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(gru_onnx_out.shape),
    buffer_ptr=gru_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['gru'].run_with_iobinding(gru_io_binding)
    onnx_results.append(gru_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        gru_pytorch_out = model.update.gru(net_t)
    pytorch_results.append(gru_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("gru", gru_onnx_out, gru_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {gru_onnx_out}')
        print(f'Pytorch Output: {gru_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("gru", max_abs, all_ok)


gru: max_abs_diff=7.299e-03  allclose=False
gru: max_abs_diff=7.299e-03  allclose=False
gru: max_abs_diff=7.299e-03  allclose=False
gru: max_abs_diff=7.299e-03  allclose=False
gru: max_abs_diff=7.299e-03  allclose=False
gru: max_abs_diff=7.299e-03  allclose=False
gru: max_abs_diff=7.299e-03  allclose=False
gru: max_abs_diff=7.299e-03  allclose=False
gru: max_abs_diff=7.299e-03  allclose=False
gru: max_abs_diff=7.299e-03  allclose=False
Onnx Output: tensor([[[-0.8348061442, -0.5173228979, -0.4597114027,  ..., -2.5187394619, -2.6054465771,  0.0030873790],
         [-0.5461102128, -0.7381848693, -0.6901669502,  ..., -2.6077523232, -2.5532159805, -0.2979394197],
         [-0.7737390399, -0.5927777886, -0.4138654470,  ..., -2.2568047047, -2.0182957649,  0.1862450540],
         ...,
         [-0.5110284686, -0.3916021883, -0.4594575465,  ..., -2.2657213211, -2.1633889675,  0.3914180100],
         [-0.7488061786, -0.0838363171, -0.4147749245,  ..., -2.1639928818, -1.9237400293,  0.3461208642]

In [15]:
#--------------------w----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
w_onnx_out = torch.empty((B, E_real, 2), device=net_t.device, dtype=torch.float32)
w_io_binding = module_sessions['w'].io_binding()
bind_torch_inputs(w_io_binding, {'net_input': net_t})

w_io_binding.bind_output(
    name='weight_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(w_onnx_out.shape),
    buffer_ptr=w_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['w'].run_with_iobinding(w_io_binding)
    onnx_results.append(w_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        w_pytorch_out = model.update.w(net_t)
    pytorch_results.append(w_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("w", w_onnx_out, w_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {w_onnx_out}')
        print(f'Pytorch Output: {w_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("w", max_abs, all_ok)


w: max_abs_diff=2.391e-04  allclose=False
w: max_abs_diff=2.391e-04  allclose=False
w: max_abs_diff=2.391e-04  allclose=False
w: max_abs_diff=2.391e-04  allclose=False
w: max_abs_diff=2.391e-04  allclose=False
w: max_abs_diff=2.391e-04  allclose=False
w: max_abs_diff=2.391e-04  allclose=False
w: max_abs_diff=2.391e-04  allclose=False
w: max_abs_diff=2.391e-04  allclose=False
w: max_abs_diff=2.391e-04  allclose=False
Onnx Output: tensor([[[0.2102435231, 0.1048217416],
         [0.1849069595, 0.1014463305],
         [0.2296333313, 0.1083137393],
         ...,
         [0.2310675979, 0.1056440473],
         [0.3373426199, 0.1222148538],
         [0.3044887781, 0.1185547113]]], device='cuda:0')
Pytorch Output: tensor([[[0.2103119344, 0.1048397645],
         [0.1848833561, 0.1014085636],
         [0.2296974808, 0.1083278805],
         ...,
         [0.2311458439, 0.1056691632],
         [0.3374431431, 0.1222286075],
         [0.3045251667, 0.1185550615]]], device='cuda:0')
All Onnx Results 

In [16]:
#--------------------d----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
d_onnx_out = torch.empty((B, E_real, 2), device=net_t.device, dtype=torch.float32)
d_io_binding = module_sessions['d'].io_binding()
bind_torch_inputs(d_io_binding, {'net_input': net_t})

d_io_binding.bind_output(
    name='delta_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(d_onnx_out.shape),
    buffer_ptr=d_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['d'].run_with_iobinding(d_io_binding)
    onnx_results.append(d_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        d_pytorch_out = model.update.d(net_t)
    pytorch_results.append(d_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("d", d_onnx_out, d_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {d_onnx_out}')
        print(f'Pytorch Output: {d_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("d", max_abs, all_ok)


d: max_abs_diff=9.190e-03  allclose=False
d: max_abs_diff=9.190e-03  allclose=False
d: max_abs_diff=9.190e-03  allclose=False
d: max_abs_diff=9.190e-03  allclose=False
d: max_abs_diff=9.190e-03  allclose=False
d: max_abs_diff=9.190e-03  allclose=False
d: max_abs_diff=9.190e-03  allclose=False
d: max_abs_diff=9.190e-03  allclose=False
d: max_abs_diff=9.190e-03  allclose=False
d: max_abs_diff=9.190e-03  allclose=False
Onnx Output: tensor([[[ 0.2083182633, -0.1112524644],
         [-0.2077078968,  0.5033736825],
         [ 0.1533717066, -0.2271837294],
         ...,
         [ 0.2115377039, -0.0031299964],
         [-0.0698670596, -0.0810295865],
         [-0.0756457821,  0.0328541175]]], device='cuda:0')
Pytorch Output: tensor([[[ 0.2082401812, -0.1113203838],
         [-0.2077959925,  0.5033258796],
         [ 0.1533305347, -0.2271624506],
         ...,
         [ 0.2114167064, -0.0031604506],
         [-0.0699268654, -0.0810475796],
         [-0.0756778121,  0.0329031497]]], device='cu

In [17]:
# Summary: max |ONNX − PyTorch| over all repeat iterations (patchify, update, then modular blocks)
comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,model,max_abs_diff,allclose
0,patchify,153.000000,False
1,update,69.585403,False
2,corr,0.003113,False
3,norm,0.000002,False
4,c1,0.005961,False
5,c2,0.005734,False
6,agg_kk,0.003576,False
7,agg_ij,0.002454,False
8,gru,0.007299,False
9,w,0.000239,False
